# **PUC-Rio | Departamento de Engenharia Industrial**
# **ENG 4560: Projeto Integrado VI - Distribuição Física**

---

# **Aula 3 — Modelagem matemática do CVRP (Parte 1)**  
**Prof. Marcello Congro (marcellocongro@puc-rio.br)**

---

## 🎯 Objetivos da Aula

Ao final desta aula, você será capaz de:

1. **Carregar** uma instância preparada na Aula 2 (`nodes.csv`, `D.npy`, `Cvar.npy`, `q.npy`, `s.npy`, `Tmov_h.npy`, `params.json`);
2. **Entender a estrutura** de um modelo de Programação Linear Inteira (PLI/MIP) no Pyomo;
3. Definir variáveis binárias de roteamento $x_{ij}$ e conectar isso a uma interpretação logística;
4. Construir a **função objetivo** com:
   - custo variável proporcional à distância ($c_{ij}$)
   - custo fixo por veículo/rota (aqui: VUC)
5. Implementar restrições fundamentais:
   - visita única (entrada/saída)
   - conservação de fluxo
   - balanço no depósito
   - capacidade **agregada** (via número de rotas)
   - jornada **agregada** (via número de rotas)
6. Resolver o modelo com um solver MIP e **interpretar**:
   - custo total
   - número de veículos (rotas)
   - estrutura das rotas retornadas
7. Diagnosticar **subtours** (ciclos desconectados do depósito), já que nesta etapa **não usamos MTZ**.

O objetivo não é apenas obter uma solução computacional, mas compreender:

- como traduzir um problema operacional em linguagem matemática;
- quais hipóteses estamos assumindo;
- quais limitações surgem quando a formulação ainda não está completa.

⚠️ Atenção: ao final desta aula você perceberá que uma solução ótima do solver nem sempre representa uma solução operacionalmente aceitável.

Essa constatação será fundamental para a Aula 4.

---

## 📌 Contexto

Na Aula 2, estruturamos uma instância contendo:

- Conjunto de nós $$N=\{0,1,\ldots,n\}$$ (0 é o depósito)
- Distâncias (km) $$D_{ij}$$
- Custos variáveis (reais) $$C_{ij}$$
- Tempos de deslocamento (h) $$T_{ij}$$
- Demandas (kg) $$q_i$$
- Tempos de atendimento (h) $$s_i$$

Nesta aula, vamos transformar isso em um **modelo matemático** que o solver consegue resolver.

---

## Como trabalhar neste notebook

Este material foi estruturado como um **estudo guiado**.

Recomenda-se:

1. Ler cada seção antes de executar as células;
2. Discutir em grupo as perguntas indicadas;
3. Anotar observações durante a execução.

Evite executar todas as células de uma vez. O aprendizado ocorre durante a construção do modelo.

# Estrutura da Aula Prática

1. Carregar e validar instância
2. Definir parâmetros logísticos
3. Construir modelo matemático em Pyomo
4. Resolver com solver MIP
5. Interpretar solução
6. Diagnosticar limitações estruturais

In [ ]:
# ============================================================
# (1) INSTALAÇÃO DO SOLVER
# ============================================================
# Nesta etapa preparamos o ambiente computacional. O solver é o responsável por resolver o problema matemático.
#
# IMPORTANTE:
# O solver NÃO entende logística. Ele apenas resolve equações matemáticas.
# Se a formulação estiver incompleta, o solver conseguirá perceber isso?

!apt-get update -y
!apt-get install -y coinor-cbc

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 3s (1,284 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+d

In [ ]:
# ============================================================
# (2) CONFIGURAÇÃO DO SOLVER GUROBI
# ============================================================
# O Gurobi é um solver comercial altamente eficiente.
# Nesta disciplina poderemos usá-lo, vinculando a licença acadêmica da PUC-Rio.

# Observe:
# Trocar o solver NÃO muda o modelo matemático.
# Apenas muda a eficiência da busca da solução.

import os
!pip install gurobipy
import gurobipy

# ============================================================
# TODO (ALUNO — EM SALA)
# 1) Substitua os valores abaixo pelas credenciais do seu grupo (WLS).
# 2) Depois, rode esta célula e confirme se a mensagem "Credenciais WLS configuradas." aparece.
# ============================================================

os.environ["GRB_WLSACCESSID"] = "PREENCHER_AQUI"    #Atualize esta linha com a sua ID
os.environ["GRB_WLSSECRET"] = "PREENCHER_AQUI"      #Atualize esta linha com a sua ID
os.environ["GRB_LICENSEID"] = "PREENCHER_AQUI"      #Atualize esta linha com a sua ID

Credenciais WLS configuradas.


In [ ]:
# ============================================================
# (3) CARREGAMENTO DA INSTÂNCIA
# ============================================================
# No projeto utilizamos dados reais de operação logística.
# Uma instância contém: clientes, demandas, distâncias, tempos de serviço.
# Sem dados confiáveis, não existe otimização confiável.
# ============================================================

!rm -f *.csv *.npy *.json
from google.colab import files
import os

print("Faça upload dos arquivos da instância:")
uploaded = files.upload()

print("Arquivos disponíveis:")
print(os.listdir())

# O que aconteceria se a matriz de distância estivesse errada? O solver perceberia?

Faça upload dos arquivos da instância:


Saving Cvar.npy to Cvar.npy
Saving D.npy to D.npy
Saving nodes.csv to nodes.csv
Saving params.json to params.json
Saving q.npy to q.npy
Saving s.npy to s.npy
Saving Tmov_h.npy to Tmov_h.npy
Arquivos disponíveis:
['.config', 'nodes.csv', 'Cvar.npy', 's.npy', 'D.npy', 'q.npy', 'Tmov_h.npy', 'params.json', 'sample_data']


In [ ]:
# ============================================================
# (4) LEITURA E VERIFICAÇÃO DOS DADOS
# ============================================================
# Aqui verificamos se os dados carregados fazem sentido. Antes de modelar, garantimos que a instância está consistente.
# Modelos MIP são altamente sensíveis a inconsistências dimensionais.
# ============================================================

import numpy as np
import pandas as pd
import json

nodes = pd.read_csv("nodes.csv")
D = np.load("D.npy")
C = np.load("Cvar.npy")
q = np.load("q.npy")
s = np.load("s.npy")

n = len(nodes)

assert D.shape == (n, n), "Dimensão incorreta de D"
assert C.shape == (n, n), "Dimensão incorreta de C"
assert q.shape == (n,), "Dimensão incorreta de q"
assert s.shape == (n,), "Dimensão incorreta de s"
assert nodes.loc[0, "id"] == 0, "Nó 0 deve ser o depósito"

print(f"Instância carregada: {n-1} clientes + depósito")

# ============================================================
# TODO (ALUNO — EM SALA)
# Exiba rapidamente:
# - demanda total (kg)
# - maior demanda de um cliente (kg)
# Isso será útil para discutir viabilidade da frota.
# ============================================================

# print("Demanda total (kg):", ? )
# print("Maior demanda (kg):", ? )

Instância carregada: 60 clientes + depósito


In [ ]:
# ============================================================
# (5) PARÂMETROS LOGÍSTICOS
# ============================================================
# Importante: Capacidade e jornada serão modeladas de forma agregada.
# Isso garante viabilidade global, mas NÃO garante que cada rota individual respeite capacidade e jornada.
# Nesta aula consideramos também frota homogênea (somente VUCs sendo utilizados)
# Essas simplificações são comuns em primeiras modelagens, mas podem esconder limitações operacionais reais.
# ============================================================

Q = 3000.0        # Capacidade do VUC (kg) (aqui estamos supondo frota homogênea)
f = 550.0         # Custo fixo por veículo (R$)
H = 8.0           # Jornada máxima (h)
v_kmh = 40.0      # Velocidade média

T = D / v_kmh

In [ ]:
# ============================================================
# (6) CONSTRUÇÃO DO MODELO
# ============================================================
# Chamamos o Pyomo e criamos a estrutura do modelo de Programação
# Linear Inteira Mista.
#============================================================

from pyomo.environ import *

model = ConcreteModel()

model.N = RangeSet(0, n-1)
model.C = RangeSet(1, n-1)

In [ ]:
# ============================================================
# (7) VARIÁVEIS DE DECISÃO
# ============================================================
# x[i,j] = 1 se o veículo percorre o arco i -> j;
# Observe: Ainda NÃO SABEMOS qual veículo faz qual rota. Isso será tratado em aulas futuras.
# Criação de apenas arcos válidos (ou seja, i diferente de j)
# ============================================================

model.A = [(i,j) for i in range(n) for j in range(n) if i != j]
model.x = Var(model.A, domain=Binary)

# ============================================================
# TODO (ALUNO — EM SALA)
# Exiba:
# - número de arcos possíveis (|A|)
# - estimativa rápida de variáveis binárias
# (isso será usado na discussão de complexidade)
# ============================================================

# print("Número de arcos (|A|):", ? )

In [ ]:
# ============================================================
# (8) NÚMERO DE VEÍCULOS UTILIZADOS
# ============================================================
# Criamos uma função para determinar o número de veículos.
# Aqui estimamos implicitamente quantos veículos serão necessários.
# Ainda não há identificação individual de veículo. Isso pode gerar situações inesperadas.
# ============================================================

def m_expr(model):
    return sum(model.x[0,j] for j in model.C)

model.m = Expression(rule=m_expr)

In [ ]:
# ============================================================
# (9) FUNÇÃO OBJETIVO
# ============================================================
# O que significa "ótimo"? = Estamos minimizando o custo de deslocamento
# Mas atenção: minimizar distâncias não garante operação realista.
# ============================================================

def obj_rule(model):
    travel_cost = sum(C[i,j] * model.x[i,j] for (i,j) in model.A)
    fixed_cost = f * model.m
    return travel_cost + fixed_cost

model.obj = Objective(rule=obj_rule, sense=minimize)

In [ ]:
# ============================================================
# (10) RESTRIÇÕES
# ============================================================
# As restrições são o "coração" do modelo. Elas informam ao solver o que "não pode acontecer".
# No nosso caso, já impedimos ciclos desconectados?
# ============================================================

# Saida única
def out_rule(model, i):
    return sum(model.x[i,j] for j in model.N if j != i) == 1

model.out_degree = Constraint(model.C, rule=out_rule)

# Entrada única
def in_rule(model, j):
    return sum(model.x[i,j] for i in model.N if i != j) == 1

model.in_degree = Constraint(model.C, rule=in_rule)

# ============================================================
# Conservação de fluxo
# OBS: A conservação de fluxo é redundante quando entrada=1 e saída=1 já são impostas.
# Mantemos por clareza estrutural.
# ============================================================

def flow_rule(model, i):
    outflow = sum(model.x[i,j] for j in model.N if j != i)
    inflow  = sum(model.x[j,i] for j in model.N if j != i)
    return outflow - inflow == 0

model.flow = Constraint(model.C, rule=flow_rule)

# ============================================================
# TODO (ALUNO — EM SALA)
# Conte quantas restrições foram criadas até aqui (aproximação).
# Dica: número de clientes = n-1.
# - out_degree: (n-1)
# - in_degree:  (n-1)
# - flow:       (n-1)
# Registre o total e discuta: isso ainda garante conectividade global?
# ============================================================

# total_restr_aprox = ?
# print("Total aproximado de restrições até aqui:", total_restr_aprox)

In [ ]:
# ============================================================
# (11) BALANÇO NO DEPÓSITO
# ============================================================
# Criação de função para garantir saída e retorno ao depósito
# Porém isso não garante uma única rota contínua.
# ============================================================

def depot_balance(model):
    return sum(model.x[0,j] for j in model.C) == \
           sum(model.x[i,0] for i in model.C)

model.depot_balance = Constraint(rule=depot_balance)

In [ ]:
# ============================================================
# (12) CAPACIDADE AGREGADA
# ============================================================
# Criação de função para que a soma das demandas não exceda a capacidade.
# Ainda estamos tratando capacidade de forma agregada
# Não sabemos qual veículo atende qual cliente
# ============================================================

def capacity_rule(model):
    total_demand = sum(q[i] for i in range(1,n))
    return total_demand <= Q * model.m

model.capacity = Constraint(rule=capacity_rule)

# ============================================================
# TODO (ALUNO — EM CASA)
# Calcule e imprima:
# 1) demanda total
# 2) capacidade total disponível = Q * m
# usando o valor de m após resolver o modelo.
# (você fará isso mais adiante, após a célula 17)
# ============================================================

In [ ]:
# ============================================================
# (13) COMPLEXIDADE
# ============================================================
# O número de variáveis cresce aproximadamente como n(n−1).
# Isso significa que dobrar clientes pode quadruplicar variáveis.
# Será que esse modelo vai rodar rapidamente para nossa maior instância (60 clientes)?
# ============================================================

print("Número aproximado de variáveis binárias:", len(model.A))
print("Número de clientes:", n-1)



Número aproximado de variáveis binárias: 3660
Número de clientes: 60


In [ ]:
# ============================================================
# (14) ESCOLHA DO SOLVER
# ============================================================
# Por padrão usamos CBC (open-source).
# Se você possui licença acadêmica da PUC-Rio:
# - Altere SOLVER_NAME para "gurobi_direct" ou "cplex"
# ============================================================

from pyomo.opt import SolverFactory

# ============================================================
# TODO (ALUNO — EM SALA)
# Escolha o solver do seu grupo:
# - "cbc" (sempre disponível)
# - "gurobi_direct" (se WLS estiver ok)
# E ative um limite de tempo de solução (time limit) se necessário.
# ============================================================

SOLVER_NAME = "cbc"  # opções: "cbc", "gurobi_direct", "cplex"

solver = SolverFactory(SOLVER_NAME)

if not solver.available():
    raise RuntimeError(f"O solver {SOLVER_NAME} não está disponível.")

# ============================================================
# TODO (ALUNO — EM SALA)
# Configure um TimeLimit (em segundos) para evitar travamentos em instâncias grandes.
# Sugestão inicial: 60 s (depois teste 120 s).
# Dica: para alguns solvers, isso pode variar. Use try/except para não quebrar.
# ============================================================

# try:
#     solver.options["TimeLimit"] = ?
# except:
#     print("Este solver não aceita a opção TimeLimit dessa forma.")

# ============================================================
# Antes de rodar a próxima célula, responda: O que você espera observar?
# ============================================================

  The solver plugin was not registered.
  Please confirm that the 'pyomo.environ' package has been imported.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyomo/opt/base/solvers.py", line 171, in __call__
    raise RuntimeError(
RuntimeError:   The solver plugin was not registered.
  Please confirm that the 'pyomo.environ' package has been imported.


ApplicationError: Solver (cbc) not available

In [ ]:
# ============================================================
# (15) RESOLUÇÃO DO MODELO
# ============================================================

import time

print(f"Resolvendo com solver: {SOLVER_NAME}")

start_time = time.time()
results = solver.solve(model)
end_time = time.time()

elapsed = end_time - start_time

print("\nStatus:", results.solver.status)
print("Termination condition:", results.solver.termination_condition)
print(f"Tempo de solução: {elapsed:.2f} segundos")

# ============================================================
# TODO (ALUNO — EM SALA)
# Imprima também:
# - tempo em minutos (com 2 casas)
# - tempo em horas (com 3 casas)
# ============================================================

# print(f"Tempo (min): {?:.2f}")
# print(f"Tempo (h): {?:.3f}")

Resolvendo com solver: gurobi_direct


GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
# ============================================================
# (16) RESULTADOS DO MODELO
# ============================================================
# O solver encontra a solução ótima?
# Uma solução ótima matematicamente significa ótima operacionalmente?
# ============================================================

print("Custo total:", value(model.obj))
print("Número de veículos utilizados:", value(model.m))

# ============================================================
# TODO (ALUNO — EM SALA)
# Registre os resultados do grupo:
# - custo total
# - m (número de veículos)
# - tempo de solução
# e compare com outra equipe (instância diferente).
# ============================================================

In [ ]:
# ============================================================
# (17) EXTRAÇÃO DOS ARCOS SELECIONADOS
# ============================================================
# Aqui observamos quais conexões foram escolhidas.
# Mas atenção: ainda não sabemos se formam rotas válidas.
# ============================================================

selected_arcs = [(i,j) for (i,j) in model.A if value(model.x[i,j]) > 0.5]

print("Total de arcos selecionados:", len(selected_arcs))
print("Alguns arcos:", selected_arcs[:20])

# ============================================================
# TODO (ALUNO — EM SALA)
# Verificação rápida:
# O número total de arcos selecionados deveria ser aproximadamente:
#   2*(n-1) + 2*m   ?  (discuta!)
# Sem subtour elimination, essa expectativa pode falhar.
# Só registre a discussão.
# ============================================================

In [ ]:
#===========================================================
# (18) RECONSTRUÇÃO DAS ROTAS
# ============================================================
# Agora conseguimos visualizar o comportamento emergente do modelo
# ============================================================

succ = {i:j for (i,j) in selected_arcs}

starts = [j for (i,j) in selected_arcs if i == 0]

routes = []

for start in starts:
    route = [0, start]
    current = start
    while current != 0:
        current = succ.get(current, None)
        if current is None:
            break
        route.append(current)
    routes.append(route)

print("\nRotas reconstruídas:")
for r in routes:
    print(r)

# ============================================================
# TODO (ALUNO — EM SALA)
# Imprima o comprimento (número de nós) de cada rota.
# Isso ajuda a perceber se existem rotas "curtas demais" ou estranhas.
# ============================================================

# for r in routes:
#     print(len(r))

print("Clientes totais:", n-1)

visited=set()

for r in routes:
    visited.update(r)

visited.discard(0)

print("Clientes atendidos via depósito:", len(visited))

In [ ]:
# ============================================================
# (19) CHECAGEM PÓS-SOLUÇÃO — JORNADA POR ROTA (VALIDAÇÃO)
# ============================================================
# Importante:
# NÃO impusemos jornada no modelo matemático.
# Portanto, após obter uma solução, precisamos VALIDAR se cada rota
# respeita ou não o limite operacional de H horas.
#
# A checagem abaixo é um "pós-processamento" (não afeta a solução do solver).
# Ela nos permite comparar: solução matemática vs. viabilidade operacional.
# ============================================================

def route_time(route):
    # route: lista do tipo [0, ..., 0]
    t_mov = 0.0
    for a in range(len(route)-1):
        i = route[a]
        j = route[a+1]
        t_mov += T[i, j]

    # tempo de serviço: soma dos atendimentos dos clientes na rota (exceto depósito)
    clients_in_route = [node for node in route if node != 0]
    t_serv = sum(s[i] for i in clients_in_route)

    return t_mov + t_serv, t_mov, t_serv

print(f"Limite operacional de jornada: H = {H:.2f} h\n")

violations = 0
for r_id, r in enumerate(routes, start=1):
    t_total, t_mov, t_serv = route_time(r)
    status = "OK" if t_total <= H + 1e-6 else "VIOLA"
    if status == "VIOLA":
        violations += 1

    print(f"Rota {r_id}: tempo total = {t_total:.2f} h (mov={t_mov:.2f}, serv={t_serv:.2f}) -> {status}")

print(f"\nTotal de rotas que violam H: {violations}")

In [ ]:
# ============================================================
# (20) DIAGNÓSTICO DE SUBTOURS
# ============================================================
# Caso existam ciclos desconectados ao depósito, isso não é erro do solver.
# É uma consequência direta da formulação do modelo matemático empregado aqui.
# ============================================================

unvisited = set(range(n))
subtours = []

while unvisited:
    node = next(iter(unvisited))
    cycle = []
    current = node
    while current in unvisited:
        unvisited.remove(current)
        cycle.append(current)
        current = succ.get(current, None)
        if current is None:
            break
    if cycle:
        subtours.append(cycle)

sub_no_depot = [c for c in subtours if 0 not in c and len(c)>1]

if sub_no_depot:
    print("Subtours encontrados:", sub_no_depot)
else:
    print("Nenhum subtour detectado.")

#Estamos apenas detectando subtours, sem corrigi-los.

# ============================================================
# TODO (ALUNO — EM SALA)
# Imprima:
# - número de subtours (sem depósito)
# - tamanho de cada subtour
# ============================================================

# print("Qtd subtours sem depósito:", ?)
# print("Tamanhos:", ?)

In [ ]:
# ============================================================
# (21) VISUALIZAÇÃO DAS ROTAS E SUBTOURS (caso existam)
# ============================================================
# Observe cuidadosamente o mapa.
# Existem rotas que não passam pelo depósito/CD?
# Discuta com sua equipe antes de continuar.
# ============================================================

import matplotlib.pyplot as plt

# Coordenadas
x_coords = nodes["lon"].values
y_coords = nodes["lat"].values

plt.figure(figsize=(8,8))

# Clientes
plt.scatter(x_coords[1:], y_coords[1:], color='blue', label='Clientes')

# Depósito
plt.scatter(x_coords[0], y_coords[0], color='red', s=150, label='Depósito')

plt.title("Nós do Problema (CVRP)")
plt.legend()
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.show()

plt.figure(figsize=(8,8))

# Plotar nós
plt.scatter(x_coords[1:], y_coords[1:], color='blue')
plt.scatter(x_coords[0], y_coords[0], color='red', s=150)

# Plotar rotas
for route in routes:
    for i in range(len(route)-1):
        i_node = route[i]
        j_node = route[i+1]

        plt.plot([x_coords[i_node], x_coords[j_node]],
                 [y_coords[i_node], y_coords[j_node]],
                 color='green')

plt.title("Rotas Reconstruídas")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.show()

plt.figure(figsize=(8,8))

# Nós
plt.scatter(x_coords[1:], y_coords[1:], color='blue')
plt.scatter(x_coords[0], y_coords[0], color='red', s=150)

# Rotas principais
for route in routes:
    for i in range(len(route)-1):
        i_node = route[i]
        j_node = route[i+1]
        plt.plot([x_coords[i_node], x_coords[j_node]],
                 [y_coords[i_node], y_coords[j_node]],
                 color='green')

# Subtours em vermelho
for cycle in sub_no_depot:
    for i in range(len(cycle)):
        i_node = cycle[i]
        j_node = cycle[(i+1) % len(cycle)]
        plt.plot([x_coords[i_node], x_coords[j_node]],
                 [y_coords[i_node], y_coords[j_node]],
                 color='red', linewidth=2)

plt.title("Diagnóstico Visual de Subtours")
plt.grid(True)
plt.show()

# ============================================================
# TODO (ALUNO — EM CASA)
# Adicione (no final do notebook) um bloco markdown respondendo:
# 1) Subtours apareceram? Quantos?
# 2) Quanto tempo o solver levou?
# 3) Você considera a solução "aceitável" do ponto de vista operacional? Por quê?
# ============================================================

Responda, com base nos resultados obtidos:

1. O solver errou ou seguiu exatamente o que formulamos/pedimos?

2. O modelo matemático representa completamente a operação logística?

3. Qual restrição parece estar faltando?

4. Como poderíamos impedir ciclos desconectados?

5. Neste modelo utilizamos capacidade e jornada agregadas. Na prática, muitas empresas utilizam restrições individuais por veículo. Quais seriam as vantagens e desvantagens dessa modelagem?

Estas perguntas serão retomadas na Aula 4.